In [1]:
import pandas as pd
import joblib
import utils.prediction_utils as pred
import pickle
from utils import dictionaries as dictionary
from dictionaries import bucket_boundaries_dict, correction_factors

from openpyxl import load_workbook, Workbook
from openpyxl.styles import Border, Side, Alignment, Font
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import math

from openpyxl.styles import PatternFill, Alignment

import matplotlib.pyplot as plt
from openpyxl.drawing.image import Image
from openpyxl import load_workbook

In [2]:
converse_dict = pickle.load(open('converse_dict.pkl', 'rb'))
# Load the mode
model = joblib.load("25-67_bereik.pkl")

[15:19:30] WARNING: C:/Jenkins/workspace/xgboost-win64_release_0.90/src/objective/regression_obj.cu:152: reg:linear is now deprecated in favor of reg:squarederror.


In [3]:
def color_residual_columns(file_path, output_path):
    # Load the workbook and select the active worksheet
    wb = load_workbook(file_path)
    ws = wb.active

    # Define colors for different thresholds
    light_red = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    medium_red = PatternFill(start_color="FF5733", end_color="FF5733", fill_type="solid")
    dark_red = PatternFill(start_color="B22222", end_color="B22222", fill_type="solid")
    light_blue = PatternFill(start_color="ADD8E6", end_color="ADD8E6", fill_type="solid")
    medium_blue = PatternFill(start_color="6495ED", end_color="6495ED", fill_type="solid")
    dark_blue = PatternFill(start_color="00008B", end_color="00008B", fill_type="solid")

    # Columns to apply coloring
    columns_to_color = ["residual_lgb", "residual_cat", "old_model_residual_corrected", "old_model_residual"]

    # Identify column indices for the specified columns
    header_row = ws[1]
    col_indices = {cell.value: idx for idx, cell in enumerate(header_row, start=1)}
    target_indices = [col_indices[col] for col in columns_to_color if col in col_indices]

    # Apply coloring
    for row in ws.iter_rows(min_row=2, max_col=max(target_indices), max_row=ws.max_row):
        for col_idx in target_indices:
            cell = row[col_idx - 1]  # Adjusting for 0-based index
            if cell.value is not None:
                if cell.value >= 8:
                    cell.fill = medium_red
                elif cell.value >= 4:
                    cell.fill = light_red
                elif cell.value <= -8:
                    cell.fill = medium_blue
                elif cell.value <= -4:
                    cell.fill = light_blue

    # Autofit width and center-align for the last 8 columns
    last_8_cols = ws.max_column - 8  # Start of the last 8 columns
    for col_idx in range(last_8_cols, ws.max_column + 1):
        max_length = 0
        for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=col_idx, max_col=col_idx):
            for cell in row:
                if cell.value is not None:
                    cell.alignment = Alignment(horizontal="center")  # Center alignment
                    max_length = max(max_length, len(str(cell.value)))

        # Set the column width
        ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = max_length + 2  # Padding for readability

    # Save the workbook
    wb.save(output_path)
    print(f"Coloring and formatting applied. File saved to {output_path}")

def prep_dataframe(merged_df):

    df = merged_df.copy()

    df['timef_olympic_period'] = df.apply(pred.calculate_total_overlap, axis=1, period_list=(dictionary.olympic_period_list))
    df['timef_football_period'] = df.apply(pred.calculate_total_overlap, axis=1, period_list=(dictionary.football_wk_ek_list))
    # Convert start_date and end_date to datetime objects and calculate required values
    df['start_date'] = pd.to_datetime(df['start_date'], format="%d%m%y", errors='coerce')
    df['end_date'] = pd.to_datetime(df['end_date'], format="%d%m%y", errors='coerce')

    df['timef_start_month'] = df['start_date'].dt.month
    df['timef_end_month'] = df['end_date'].dt.month
    df['timef_start_year'] = df['start_date'].dt.year
    df['timef_end_year'] = df['end_date'].dt.year
    df['timef_start_week'] = df['start_date'].dt.strftime("%V").astype(int)
    df['timef_end_week'] = df['end_date'].dt.strftime("%V").astype(int)
    df['timef_start_day'] = df['start_date'].dt.day
    df['timef_end_day'] = df['end_date'].dt.day
    df['timef_duration_days'] = df.apply(lambda row: (row['end_date'] - row['start_date']).days + 1, axis=1)

    df['timef_spreiding'] = 0.75

    df['meta_hoofdspot_duration'] = 30
    df['meta_second_spot_duration'] = 0
    df['meta_tagon'] = False

    df['meta_alcohol_adv'] = df['alcohol_kansspelen']
    df['meta_kansspelen_adv'] = df['alcohol_kansspelen']
    df['NPO_1_primetime_GRP'] = df['npo1_grp_primetime']
    df['NPO_1_daynight_GRP'] = df['npo1_grp_daynight']
    df['NPO_2_primetime_GRP'] = df['npo2_grp_primetime']
    df['NPO_2_daynight_GRP'] = df['npo2_grp_daynight']
    df['NPO_3_primetime_GRP'] = df['npo3_grp_primetime']
    df['NPO_3_daynight_GRP'] = df['npo3_grp_daynight']

    df = df[['timef_olympic_period', 'timef_football_period', 'timef_start_year',
            'timef_end_year', 'timef_start_month', 'timef_end_month',
            'timef_start_week', 'timef_end_week', 'timef_start_day',
            'timef_end_day', 'timef_duration_days', 'timef_spreiding',
            'meta_hoofdspot_duration', 'meta_second_spot_duration', 'meta_tagon',
            'meta_alcohol_adv', 'meta_kansspelen_adv', 'NPO_1_primetime_GRP',
            'NPO_1_daynight_GRP', 'NPO_2_primetime_GRP', 'NPO_2_daynight_GRP',
            'NPO_3_primetime_GRP', 'NPO_3_daynight_GRP']]

    return df

def correction_bereik(tg, grp):
    def get_bucket(value, bucket_ranges):
        for i, bucket_range in enumerate(bucket_ranges):
            if value <= bucket_range:
                return i
        return len(bucket_ranges)

    # Get the bucket ranges for the target
    bucket_ranges = bucket_boundaries_dict.get(tg)

    if bucket_ranges:
        # Get the bucket index for the value
        bucket_index = get_bucket(grp, bucket_ranges)

        if tg in correction_factors and bucket_index in correction_factors[tg]:
            correction = correction_factors[tg][bucket_index]
            return correction
        else:
            # Return 0.8 if correction factors are not found
            return 0.8
    else:
        # Return 0.8 if target group doesn't exist
        return 0.8

def write_metrics_to_sheet(file_path, output_path, df):
    # Load the workbook
    wb = load_workbook(file_path)

    # Create Sheet 2 if it doesn't exist
    if 'Sheet2' not in wb.sheetnames:
        wb.create_sheet('Sheet2')
    ws = wb['Sheet2']

    # Clear Sheet2 for fresh metrics
    for row in ws.iter_rows():
        for cell in row:
            cell.value = None

    # Metrics to compute
    metrics = ['MSE', 'RMSE', 'R²', 'MAE']

    # Model names and their residuals
    models = {
        'LightGBM': 'prediction_lgb',
        'CatBoost': 'prediction_cat',
        'Old Model': 'prediction_old_model',
        'Corrected Old Model': 'corrected_prediction_old_model'
    }

    # Start writing tables
    start_row = 1
    for model_name, residual_col in models.items():
        # Compute metrics
        mse = mean_squared_error(df['ster_1+_total'], df[residual_col])
        rmse = math.sqrt(mse)
        r2 = r2_score(df['ster_1+_total'], df[residual_col])
        mae = mean_absolute_error(df['ster_1+_total'], df[residual_col])

        # Prepare data for table
        data = [
            ['Metric', 'Value'],
            ['MSE', round(mse, 2)],
            ['RMSE', round(rmse, 2)],
            ['R²', round(r2, 2)],
            ['MAE', round(mae, 2)]
        ]

        # Write table to Sheet2
        ws.cell(row=start_row, column=1, value=model_name).font = Font(bold=True, size=14)
        start_row += 1  # Leave a row for the title
        for i, row_data in enumerate(data):
            for j, value in enumerate(row_data):
                cell = ws.cell(row=start_row + i, column=j + 1, value=value)
                cell.alignment = Alignment(horizontal="center", vertical="center")

                # Add border to the cell
                thin_border = Border(
                    left=Side(style="thin"),
                    right=Side(style="thin"),
                    top=Side(style="thin"),
                    bottom=Side(style="thin")
                )
                cell.border = thin_border

        # Leave some space between tables
        start_row += len(data) + 12

    # Save the workbook
    wb.save(output_path)
    print(f"Metrics written to Sheet2 and saved in {output_path}")

def add_residual_plots_to_sheet(file_path, output_path, df):
    # Load the workbook
    wb = load_workbook(file_path)

    # Ensure Sheet2 exists
    if 'Sheet2' not in wb.sheetnames:
        wb.create_sheet('Sheet2')
    ws = wb['Sheet2']

    # Residual columns for each model
    models = {
        'LightGBM': ('prediction_lgb', 'residual_lgb'),
        'CatBoost': ('prediction_cat', 'residual_cat'),
        'Old Model': ('prediction_old_model', 'old_model_residual'),
        'Corrected Old Model': ('corrected_prediction_old_model', 'old_model_residual_corrected')
    }

    # Generate and save residual plots
    plot_files = []
    for i, (model_name, (prediction_col, residual_col)) in enumerate(models.items()):
        plt.figure(figsize=(6, 4))
        plt.scatter(df[prediction_col], df[residual_col], alpha=0.6)
        plt.axhline(0, color='red', linestyle='--', linewidth=1)
        plt.title(f'{model_name} Residual Plot')
        plt.xlabel('Predicted Values')
        plt.ylabel('Residuals')

        # Set the axis limits
        plt.xlim(0, 60)  # X-axis limits
        plt.ylim(-15, 15)  # Y-axis limits

        plt.tight_layout()
        # Save the plot
        plot_path = f'residual_plot_{model_name}.png'
        plt.savefig(plot_path, dpi=100)
        plt.close()
        plot_files.append(plot_path)

    # Insert plots into Sheet2
    row = 1 # Start row for plots
    col = 4  # Start column for plots
    for plot_file, model_name in zip(plot_files, models.keys()):
        img = Image(plot_file)
        img.width, img.height = 400, 300  # Resize the image for better fit
        cell = ws.cell(row=row, column=col, value=model_name)
        cell.font = Font(bold=True)
        ws.add_image(img, ws.cell(row=row + 1, column=col).coordinate)
        row += 16  # Space between plots

    # Save the workbook
    wb.save(output_path)
    print(f"Residual plots added to Sheet2 and saved in {output_path}")


In [6]:
df1_lgb = pd.read_excel("test_predictions_lgb_25-67.xlsx")
df2_cat = pd.read_excel("test_predictions_cat_25-67.xlsx")[['flight_id', 'prediction_cat', 'residual_cat']]

# Merge all DataFrames on 'flight_id'
merged_df = (df1_lgb
    .merge(df2_cat, on='flight_id', how='left')
)

In [7]:
df = prep_dataframe(merged_df)

In [8]:
df['prediction'] = model.predict(df)

In [9]:
columns_to_sum = ['NPO_1_primetime_GRP', 'NPO_1_daynight_GRP', 'NPO_2_primetime_GRP', 'NPO_2_daynight_GRP', 'NPO_3_primetime_GRP', 'NPO_3_daynight_GRP']
df['total_grp'] = df[columns_to_sum].sum(axis=1)
df['correction'] = df.apply(lambda row: correction_bereik('25-67', row['total_grp']), axis=1)
df['prediction_corrected'] = (df['prediction'] * df['correction']).round(1)

merged_df['prediction_old_model'] = (df['prediction']).round(1)
merged_df['old_model_residual'] = merged_df['ster_1+_total'] - merged_df['prediction_old_model']
merged_df['corrected_prediction_old_model'] = df['prediction_corrected']
merged_df['old_model_residual_corrected'] = merged_df['ster_1+_total'] - merged_df['corrected_prediction_old_model']

merged_df = merged_df.round(1)

In [10]:
merged_df.to_excel("test_predictions_total.xlsx", index=False)

In [11]:
color_residual_columns("test_predictions_total.xlsx", "test_predictions_total.xlsx")

Coloring and formatting applied. File saved to test_predictions_total.xlsx


In [13]:
write_metrics_to_sheet("test_predictions_total.xlsx", "test_predictions_total_metrics_2.xlsx", merged_df)

Metrics written to Sheet2 and saved in test_predictions_total_metrics_2.xlsx


In [14]:
add_residual_plots_to_sheet("test_predictions_total_metrics_2.xlsx", "test_predictions_total_metrics_2.xlsx", merged_df)

Residual plots added to Sheet2 and saved in test_predictions_total_metrics_2.xlsx
